# Rotation Sensitivity Analysis

This notebook analyses how [cp_measure](https://github.com/afermg/cp_measure)
morphological features vary as a function of image rotation — a key question
for understanding which CellProfiler-compatible features are truly
rotation-invariant and which are artefacts of image orientation.

## Dataset

`rotation_dataset.ome.parquet` contains **115,200 measurements**:

| Dimension | Values | Count |
|-----------|--------|-------|
| Cells | 80 parameter configs × 20 random seeds | 1,600 |
| Angles | 0°, 5°, 10°, …, 355° | 72 |
| **Total rows** | 1,600 × 72 | **115,200** |

**Parameter grid:**

| Parameter | Values |
|-----------|--------|
| Mask shape | `EllipseMask` |
| Aspect ratio (x/y) | 1.0, 1.5, 2.0, 3.0, 4.0 |
| Cell size (y_radius px) | 60, 100, 150, 200 |
| Stain | `SpatialStain(σ=5)`, `SpatialStain(σ=20)`, `SpatialStain(σ=60)`, `ConstantStain` |
| Seeds per config | 20 (controls stain noise) |

Each row stores the 128×128 intensity image and segmentation mask as
[OME-Arrow](https://github.com/wayscience/ome-arrow) structs alongside
~372 cp_measure feature columns.

**Prerequisites:** run `simulate_rotation_sweep.py` first to generate the dataset.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from cytodataframe import CytoDataFrame

DATASET = Path("rotation_dataset.ome.parquet")
assert DATASET.exists(), f"Run simulate_rotation_sweep.py first — {DATASET} not found"

META_COLS  = ["cell_id", "angle_deg", "aspect_ratio", "y_radius",
              "stain_type", "stain_corr", "seed"]
IMAGE_COLS = ["image", "mask"]

## 1 — Dataset at a glance

In [ ]:
# Schema and size summary
schema = pq.read_schema(DATASET)
feature_cols = [f.name for f in schema
                if f.name not in META_COLS + IMAGE_COLS]

size_mb = DATASET.stat().st_size / 1e6

print(f"File:           {DATASET.name}")
print(f"Size:           {size_mb:.0f} MB (ZSTD-compressed Parquet)")
print(f"Total rows:     115,200  (1,600 cells × 72 angles)")
print(f"Total columns:  {len(schema)}")
print(f"  Metadata:     {len(META_COLS)}  ({', '.join(META_COLS)})")
print(f"  Image/mask:   {len(IMAGE_COLS)}  (OME-Arrow structs, 128×128 px)")
print(f"  Features:     {len(feature_cols)}  (cp_measure, float32)")

In [ ]:
# Parameter grid coverage — how cells are distributed
meta = pq.read_table(DATASET, columns=META_COLS + ["cell_id"]).to_pandas()
at_zero = meta[meta["angle_deg"] == 0].copy()  # one row per cell

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Aspect ratio × y_radius heatmap
pivot = at_zero.groupby(["aspect_ratio", "y_radius"]).size().unstack()
axes[0].imshow(pivot.values, aspect="auto", cmap="Blues")
axes[0].set_xticks(range(len(pivot.columns)))
axes[0].set_xticklabels(pivot.columns)
axes[0].set_yticks(range(len(pivot.index)))
axes[0].set_yticklabels(pivot.index)
axes[0].set_xlabel("y_radius (px)")
axes[0].set_ylabel("Aspect ratio")
axes[0].set_title(f"Cells per config\n(each = {pivot.values[0,0]} seeds)")
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        axes[0].text(j, i, str(pivot.values[i, j]), ha="center", va="center", fontsize=9)

# Stain type distribution
stain_counts = at_zero.groupby(["stain_type", "stain_corr"]).size()
stain_counts.plot(kind="bar", ax=axes[1], color="steelblue", edgecolor="none")
axes[1].set_xlabel("(stain_type, σ)")
axes[1].set_ylabel("Number of cells")
axes[1].set_title("Cells per stain configuration")
axes[1].tick_params(axis="x", rotation=30)

# Angle distribution
angles = sorted(meta["angle_deg"].unique())
axes[2].bar(angles, [1] * len(angles), width=4, color="coral", edgecolor="none")
axes[2].set_xlabel("Rotation angle (°)")
axes[2].set_ylabel("")
axes[2].set_title(f"{len(angles)} angles  ×  1,600 cells  =  {len(angles)*1600:,} rows")
axes[2].set_yticks([])

plt.suptitle("Rotation sweep — parameter coverage", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 2 — Cells with images via CytoDataFrame

[CytoDataFrame](https://github.com/cytomining/CytoDataFrame) extends pandas to
display OME-Arrow struct columns as inline images.  We load a small sample
(one row per aspect ratio, at angle 0°) to show the image + mask alongside features.

In [ ]:
# Load 5 representative rows (one per aspect ratio) at angle=0, with images
representative_cell_ids = (
    at_zero[at_zero["stain_type"] == "spatial"]
    .drop_duplicates("aspect_ratio")
    .sort_values("aspect_ratio")["cell_id"]
    .tolist()[:5]
)

sample_table = pq.read_table(
    DATASET,
    filters=[
        ("cell_id", "in", representative_cell_ids),
        ("angle_deg", "=", 0.0),
    ],
)
sample_df = sample_table.to_pandas()

# Build a tidy subset: metadata + image/mask + a few key features
preview_cols = META_COLS + IMAGE_COLS + ["Area", "Eccentricity", "MajorAxisLength",
                                          "Intensity_MeanIntensity", "Orientation"]
sample_df = sample_df[preview_cols].sort_values("aspect_ratio").reset_index(drop=True)

# CytoDataFrame auto-detects image/mask columns and renders them inline
CytoDataFrame(sample_df)

In [ ]:
# Show same cell at multiple rotation angles using matplotlib
from ome_arrow import OMEArrow
import pyarrow as pa

example_id = representative_cell_ids[2]  # aspect_ratio=2.0
angles_to_show = [0, 45, 90, 135, 180, 225, 270, 315]

rows = pq.read_table(
    DATASET,
    filters=[
        ("cell_id", "=", example_id),
        ("angle_deg", "in", [float(a) for a in angles_to_show]),
    ],
    columns=["angle_deg", "image", "mask"],
).to_pandas().sort_values("angle_deg")

fig, axes = plt.subplots(2, len(angles_to_show), figsize=(16, 4))
for col, (_, row) in enumerate(rows.iterrows()):
    img_arr = OMEArrow(row["image"]).export(how="numpy").squeeze()
    msk_arr = OMEArrow(row["mask"]).export(how="numpy").squeeze()
    axes[0, col].imshow(img_arr, cmap="gray", vmin=0, vmax=255)
    axes[0, col].set_title(f"{int(row['angle_deg'])}°", fontsize=9)
    axes[0, col].axis("off")
    axes[1, col].imshow(msk_arr, cmap="Greens", vmin=0, vmax=1)
    axes[1, col].axis("off")

axes[0, 0].set_ylabel("Image", fontsize=9)
axes[1, 0].set_ylabel("Mask", fontsize=9)
plt.suptitle(f"Cell {example_id} (aspect ratio 2.0) across 8 rotation angles",
             fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Show one row per stain type at the same angle to illustrate stain diversity
stain_examples = (
    at_zero.drop_duplicates("stain_type")
    .sort_values("stain_type")["cell_id"]
    .tolist()
)
# Also grab the three spatial corr levels
spatial_examples = (
    at_zero[at_zero["stain_type"] == "spatial"]
    .drop_duplicates("stain_corr")
    .sort_values("stain_corr")[["cell_id", "stain_corr"]]
)
stain_rows = pq.read_table(
    DATASET,
    filters=[
        ("cell_id", "in", spatial_examples["cell_id"].tolist() + 
                         at_zero[at_zero["stain_type"]=="constant"]["cell_id"].head(1).tolist()),
        ("angle_deg", "=", 0.0),
    ],
    columns=["cell_id", "stain_type", "stain_corr", "image"],
).to_pandas()
stain_rows = stain_rows.drop_duplicates(subset=["stain_type","stain_corr"]).sort_values(["stain_type","stain_corr"])

labels = [f"{'Constant' if r.stain_type=='constant' else f'Spatial σ={int(r.stain_corr)}'}" 
          for _, r in stain_rows.iterrows()]

fig, axes = plt.subplots(1, len(stain_rows), figsize=(12, 3))
for ax, (_, row), label in zip(axes, stain_rows.iterrows(), labels):
    img_arr = OMEArrow(row["image"]).export(how="numpy").squeeze()
    ax.imshow(img_arr, cmap="gray", vmin=0, vmax=255)
    ax.set_title(label, fontsize=10)
    ax.axis("off")

plt.suptitle("Stain type comparison (same ellipse shape, angle=0°)", fontsize=11)
plt.tight_layout()
plt.show()

## 3 — Feature stability across rotation

For each feature, compute the **coefficient of variation (CV)** across the 72
rotation angles, averaged over all 1,600 cells.  Low CV → rotation-stable;
high CV → rotation-sensitive.

In [ ]:
# Load feature columns only (skip OME-Arrow image/mask structs for speed)
df = pq.read_table(DATASET, columns=META_COLS + feature_cols).to_pandas()
print(f"Rows: {len(df):,}  |  Feature columns: {len(feature_cols)}")
df.head(3)

In [ ]:
# CV per (cell_id, feature) — std/mean across angles
grp = df.groupby("cell_id")[feature_cols]
cell_cv = grp.std() / grp.mean().abs().replace(0, np.nan)

# Mean CV across all cells
mean_cv = cell_cv.mean().sort_values(ascending=False)

print("Most rotation-SENSITIVE features (highest mean CV):")
print(mean_cv.head(10).to_string())
print()
print("Most rotation-STABLE features (lowest mean CV):")
print(mean_cv.tail(10).to_string())

In [ ]:
# Distribution of CVs
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(mean_cv.dropna(), bins=60, edgecolor="none", color="steelblue", alpha=0.8)
ax.axvline(0.05, color="orange", linestyle="--", label="CV = 0.05")
ax.set_xlabel("Mean CV across rotation angles")
ax.set_ylabel("Number of features")
ax.set_title("Distribution of rotation sensitivity (CV) across ~372 features")
ax.legend()
plt.tight_layout()
plt.show()

n_stable = (mean_cv < 0.05).sum()
print(f"Features with CV < 0.05 (essentially rotation-stable): {n_stable}/{len(mean_cv)}")

## 4 — Rotation profiles for representative features

In [ ]:
# Pick one cell from each stain type to illustrate profiles
example_cells = (
    df.groupby("stain_type", observed=True).apply(lambda g: g["cell_id"].iloc[0])
    .to_dict()
)

# Top-5 most sensitive and top-5 most stable features
top_sensitive = mean_cv.head(5).index.tolist()
top_stable    = mean_cv.tail(5).index.tolist()

fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for row_i, (features, title) in enumerate([
    (top_sensitive, "Most sensitive"),
    (top_stable,    "Most stable"),
]):
    for col_i, feat in enumerate(features):
        ax = axes[row_i, col_i]
        for stain_type, cell_id in example_cells.items():
            sub = df[df["cell_id"] == cell_id].sort_values("angle_deg")
            ax.plot(sub["angle_deg"], sub[feat], label=stain_type, linewidth=1)
        ax.set_title(feat[:28], fontsize=8)
        ax.set_xlabel("Angle (°)", fontsize=7)
        if col_i == 0:
            ax.set_ylabel(title, fontsize=8)
        ax.tick_params(labelsize=7)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right", fontsize=8, title="stain_type")
plt.suptitle("Feature value vs. rotation angle", y=1.01)
plt.tight_layout()
plt.show()

## 5 — Sensitivity by cell parameter

In [ ]:
# Does rotation sensitivity depend on aspect ratio or stain type?
# For each (cell, top-sensitive-feature), compute per-cell CV

top_feat = mean_cv.index[0]  # single most sensitive feature
per_cell = (
    df.groupby(["cell_id", "aspect_ratio", "stain_type", "stain_corr", "y_radius"])[top_feat]
    .agg(cv=lambda x: x.std() / abs(x.mean()) if abs(x.mean()) > 0 else np.nan)
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col in zip(axes, ["aspect_ratio", "stain_type"]):
    per_cell.boxplot(column="cv", by=col, ax=ax, grid=False)
    ax.set_title(f"CV of `{top_feat[:30]}` by {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("CV")

plt.suptitle("")
plt.tight_layout()
plt.show()